## Calculate score for each node in graph:

In [2]:
import pandas as pd
import pickle

with open(r"data\intermediate\nearest_categories.pkl", "rb") as f:
    nearest_categories: dict[str, pd.DataFrame] = pickle.load(f)

for category, df in nearest_categories.items():
    display(f"category: {category}", df.describe(), df.head())


'category: education'

,dist
count,7.091188e+06
mean,1.249138e+03
std,4.797710e+02
min,0.000000e+00
25%,8.609340e+02
50%,1.601000e+03
75%,1.601000e+03
max,1.601000e+03


,dist
0,697.838989
1,682.875000
2,653.182983
3,580.833008
4,575.544006


'category: leisure'

,dist
count,7.091188e+06
mean,1.044330e+03
std,5.785901e+02
min,0.000000e+00
25%,4.636690e+02
50%,1.223710e+03
75%,1.601000e+03
max,1.601000e+03


...

,dist
count,7.091188e+06
mean,1.508119e+03
std,2.798876e+02
min,0.000000e+00
25%,1.601000e+03
50%,1.601000e+03
75%,1.601000e+03
max,1.601000e+03


,dist
0,612.263000
1,627.226990
2,656.919006
3,729.268982
4,734.557983


'category: culture'

,dist
count,7.091188e+06
mean,1.482868e+03
std,3.161953e+02
min,0.000000e+00
25%,1.601000e+03
50%,1.601000e+03
75%,1.601000e+03
max,1.601000e+03


,dist
0,732.156006
1,747.119995
2,776.812012
3,849.161987
4,854.450989


In [3]:
import pandana as pdna

network = pdna.Network.from_hdf5(r"data\intermediate\denmark.backup")

In [4]:
# Check what nodes in the network have categories within max_dist
dist_df = pd.concat(
    [df["dist"] for df in nearest_categories.values()], 
    axis=1
)

max_dist = 1600
nodes = network.nodes_df

nodes["score"] = (dist_df <= max_dist).sum(axis=1)
valid_nodes = nodes[nodes["score"] > 0]

display(valid_nodes.head(), len(valid_nodes.index))

,x,y,score
0,12.072963,55.634530,5
1,12.072729,55.634554,5
2,12.072263,55.634602,5
3,12.071127,55.634711,5
4,12.071045,55.634700,5


4585598

In [ ]:
import geopandas as gpd

gdf = gpd.GeoDataFrame(
    data=valid_nodes["score"],
    geometry=gpd.points_from_xy(valid_nodes["x"], valid_nodes["y"]),
    crs=4326
)

gdf.to_file("accessibility-full.json", driver="GeoJSON")

## Option 1: Merge nodes within same grid and average score:

In [ ]:
import pygeohash as pgh

# Compute hash for all nodes/points
valid_nodes["hash"] = [
    pgh.encode(latitude=lat, longitude=lon, precision=6)
    for lat, lon in zip(valid_nodes["y"], valid_nodes["x"])
]

# Merge all rows with colliding hashes and average their scores
grouped = valid_nodes \
    .groupby("hash", as_index=False) \
    .agg({"score": "mean"})

# Make x and y column be the decoded hash
decoded = grouped["hash"].apply(pgh.decode)
grouped["x"] = decoded.str[1]
grouped["y"] = decoded.str[0]

display(grouped.head(), grouped.describe())

In [ ]:
import geopandas as gpd

gdf = gpd.GeoDataFrame(
    data={"score": grouped["score"], "hash": grouped["hash"]},
    geometry=gpd.points_from_xy(grouped["x"], grouped["y"]),
    crs=4326
)

gdf.to_file("accessibility-full-grouped6.json", driver="GeoJSON")

#gdf.explore()

## Option 2: Assign score to edges:

In [5]:
display(nodes.head(), len(nodes.index), network.edges_df.head(), len(network.edges_df.index))

,x,y,score
0,12.072963,55.634530,5
1,12.072729,55.634554,5
2,12.072263,55.634602,5
3,12.071127,55.634711,5
4,12.071045,55.634700,5


7091188

,from,to,dist
0,0,1,14.964623
1,1,2,29.692772
2,2,3,72.350900
3,3,4,5.289233
4,4,5,30.094357


7674433

In [6]:
from shapely.geometry import LineString

edges = network.edges_df.merge(
    nodes[["x", "y", "score"]],
    left_on="from",
    right_index=True,
).rename(columns={"x": "x_from", "y": "y_from", "score": "score_from"})

edges = edges.merge(
    nodes[["x", "y", "score"]],
    left_on="to",
    right_index=True,
).rename(columns={"x": "x_to", "y": "y_to", "score": "score_to"})

edges["linestring"] = edges.apply(
    lambda row: LineString([
        (row["x_from"], row["y_from"]),
        (row["x_to"], row["y_to"]),
    ]),
    axis=1
)

edges = edges[["linestring", "score_from", "score_to"]]
edges = edges[(edges["score_from"] > 0) | (edges["score_to"] > 0)]

display(edges.head(), len(edges.index))

,linestring,score_from,score_to
0,"LINESTRING (12.072963 55.6345298, 12.0727285 5...",5,5
1,"LINESTRING (12.0727285 55.6345541, 12.0722632 ...",5,5
2,"LINESTRING (12.0722632 55.6346023, 12.0711266 ...",5,5
3,"LINESTRING (12.0711266 55.6347107, 12.0710447 ...",5,5
4,"LINESTRING (12.0710447 55.6346995, 12.0705787 ...",5,5


5096371

In [ ]:
import geopandas as gpd

gdf = gpd.GeoDataFrame(edges, geometry="linestring", crs=4326)
gdf.to_file("edges.json", driver="GeoJSON")

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

engine = create_engine(os.environ["DATABASE_URL"])
gdf.to_postgis("edges", engine, if_exists="replace")

ModuleNotFoundError: No module named 'psycopg2'